<a href="https://colab.research.google.com/github/adenikeadewumi/Python-programming-for-ML-WIEOAU/blob/main/14_ml_algorithms/14_ml_algorithms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 14 - Core ML Algorithms

Algorithms: Linear Regression, Logistic Regression, Decision Trees, Random Forest, SVM, KNN
---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, r2_score
from sklearn.pipeline import Pipeline
np.random.seed(42)

X_clf, y_clf = make_classification(n_samples=1000, n_features=10, n_informative=6, random_state=42)
X_reg, y_reg = make_regression(n_samples=1000, n_features=8, noise=20, random_state=42)

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
print('Ready')

## 14.1 Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error

for name, Model, kwargs in [
    ('Linear', LinearRegression, {}),
    ('Ridge (L2)', Ridge, {'alpha': 1.0}),
    ('Lasso (L1)', Lasso, {'alpha': 0.1}),
]:
    pipe = Pipeline([('s', StandardScaler()), ('m', Model(**kwargs))])
    pipe.fit(X_tr_r, y_tr_r)
    r2 = r2_score(y_te_r, pipe.predict(X_te_r))
    print(f'{name:<18} R2 = {r2:.3f}')

## 14.2 Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
pipe = Pipeline([('s', StandardScaler()), ('m', LogisticRegression(max_iter=500))])
pipe.fit(X_tr_c, y_tr_c)
print(f'Logistic Regression accuracy: {accuracy_score(y_te_c, pipe.predict(X_te_c)):.3f}')

## 14.3 Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_tr_c, y_tr_c)
print(f'Decision Tree accuracy: {accuracy_score(y_te_c, dt.predict(X_te_c)):.3f}')

import pandas as pd
feat_imp = pd.Series(dt.feature_importances_, index=[f'F{i}' for i in range(10)])
feat_imp.sort_values().plot(kind='barh', figsize=(8, 4), title='Feature Importances')
plt.tight_layout()
plt.show()

## 14.4 Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)
rf.fit(X_tr_c, y_tr_c)
print(f'Random Forest accuracy: {accuracy_score(y_te_c, rf.predict(X_te_c)):.3f}')
print('Random Forest = many Decision Trees on random data subsets')
print('Final prediction: majority vote across all trees')

## 14.5 SVM

In [ ]:
from sklearn.svm import SVC
pipe_svm = Pipeline([('s', StandardScaler()), ('m', SVC(kernel='rbf', C=1.0))])
pipe_svm.fit(X_tr_c, y_tr_c)
print(f'SVM (RBF) accuracy: {accuracy_score(y_te_c, pipe_svm.predict(X_te_c)):.3f}')

## 14.6 KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
pipe_knn = Pipeline([('s', StandardScaler()), ('m', KNeighborsClassifier(n_neighbors=5))])
pipe_knn.fit(X_tr_c, y_tr_c)
print(f'KNN (k=5) accuracy: {accuracy_score(y_te_c, pipe_knn.predict(X_te_c)):.3f}')

## 14.7 Model Comparison

In [ ]:
models = {
    'Logistic Regression': Pipeline([('s', StandardScaler()), ('m', LogisticRegression(max_iter=500))]),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'SVM': Pipeline([('s', StandardScaler()), ('m', SVC(kernel='rbf'))]),
    'KNN': Pipeline([('s', StandardScaler()), ('m', KNeighborsClassifier(n_neighbors=5))]),
}

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_clf, y_clf, cv=5, scoring='accuracy')
    results[name] = scores.mean()
    print(f'{name:<25} {scores.mean():.3f} +/- {scores.std():.3f}')

plt.barh(list(results.keys()), list(results.values()), color='steelblue')
plt.xlim(0.5, 1.0)
plt.xlabel('CV Accuracy')
plt.title('Model Comparison (5-fold CV)')
plt.tight_layout()
plt.savefig('/tmp/model_comparison.png', dpi=80)
plt.show()

## 14.8 Hyperparameter Tuning with GridSearchCV

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, None],
}
gs = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1),
                  param_grid, cv=3, scoring='accuracy', n_jobs=-1)
gs.fit(X_tr_c, y_tr_c)
print('Best params:', gs.best_params_)
print(f'Best CV score: {gs.best_score_:.3f}')
print(f'Test accuracy: {accuracy_score(y_te_c, gs.predict(X_te_c)):.3f}')

---
## Algorithm Cheat Sheet

| Algorithm | Good For | Pros | Cons |
|-----------|----------|------|------|
| Linear Regression | Regression | Fast, interpretable | Assumes linearity |
| Logistic Regression | Classification | Fast, probabilistic | Linear boundary |
| Decision Tree | Both | Interpretable | Overfits |
| Random Forest | Both | High accuracy, robust | Less interpretable |
| SVM | Classification | Effective in high-dim | Slow on large data |
| KNN | Both | Simple | Slow inference |

## Next: [15 - Deep Learning](../15_deep_learning_intro/15_deep_learning_intro.ipynb)